<a href="https://colab.research.google.com/github/MariiaYarmolenko/HW-Data-Loves/blob/main/HW_15_4_%D0%90%D0%BD%D0%B0%D0%BB%D1%96%D0%B7_%D0%90_%D0%92_%D1%82%D0%B5%D1%81%D1%82%D1%96%D0%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [5]:
effect_size = sms.proportion_effectsize(0.20, 0.19)

In [6]:
effect_size

np.float64(0.025241594409087353)

In [7]:
required_n = sms.NormalIndPower().solve_power(
    effect_size,
    power=0.8,
    alpha=0.05,
    ratio=1
    )
required_n = ceil(required_n)

print(required_n)

24638


2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [8]:
df = pd.read_csv('/content/drive/MyDrive/Навчання DA/statistical_hypothesis/statistical_hypothesis/cookie_cats.csv')

df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [9]:
retention_stats = df.groupby('version')['retention_7'].mean()

print(retention_stats)

version
gate_30    0.190201
gate_40    0.182000
Name: retention_7, dtype: float64


$H_0: p_{30} = p_{40}$ (немає різниці між версіями)

$H_a: p_{30} > p_{40}$ (перешкода на 30 рівні краще утримує гравців)

3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [10]:
gate_30 = df[df['version'] == 'gate_30'].sample(n=required_n, random_state=22)
gate_40 = df[df['version'] == 'gate_40'].sample(n=required_n, random_state=22)

ab_test = pd.concat([gate_30, gate_40], axis=0)
ab_test.reset_index(drop=True, inplace=True)

In [13]:
conversion_rates = ab_test.groupby('version')['retention_7']
conversion_rates = conversion_rates.agg(['mean', 'std', stats.sem])
conversion_rates.columns = ['conversion_rate', 'std_deviation', 'std_error']


conversion_rates.style.format('{:.5f}')

,conversion_rate,std_deviation,std_error
version,,,
gate_30,0.18914,0.39163,0.00249
gate_40,0.17798,0.38250,0.00244


In [22]:
successes = [gate_30['retention_7'].sum(), gate_40['retention_7'].sum()]
nobs = [gate_30['retention_7'].count(), gate_40['retention_7'].count()]

In [19]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

In [25]:
z_stat, pval = proportions_ztest(successes, nobs=nobs, alternative='larger')

lower, upper = proportion_confint(successes, nobs=nobs, alpha=0.05)

print(f'z statistic: {z_stat:.4f}')
print(f'p-value: {pval:.4f}')
print(f'Довірчий інтервал 95% для групи gate_30: [{lower[0]:.4f}, {upper[0]:.4f}]')
print(f'Довірчий інтервал 95% для групи gate_40: [{lower[1]:.4f}, {upper[1]:.4f}]')

z statistic: 3.2001
p-value: 0.0007
Довірчий інтервал 95% для групи gate_30: [0.1842, 0.1940]
Довірчий інтервал 95% для групи gate_40: [0.1732, 0.1828]


Відхиляємо нульову гіпотезу, оскільки різниця є статистично значущою $p\text{-value} = 0.0007$, що значно менше за встановлений рівень значущості $\alpha = 0.05$.

Довірчі інтервали не перетинаються, верхня межа для групи $\text{gate_40} = 0.1828$, а нижня межа для групи $\text{gate_30} = 0.1842$. Відсутність перетину вказує на значущість різниці.

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


In [30]:
from scipy.stats import chi2_contingency

In [31]:
table = pd.crosstab(df['version'], df['retention_7'])

print(table)

retention_7  False  True 
version                  
gate_30      36198   8502
gate_40      37210   8279


In [32]:
chi2, p, dof, expected = chi2_contingency(table)

print(f"Статистика Хі-квадрат: {chi2:.4f}")
print(f"p-значення: {p:.4f}")

alpha = 0.05
if p < alpha:
    print(f"Висновок: Оскільки p < {alpha}, ми відхиляємо H0. Залежність між версією та утриманням є.")
else:
    print(f"Висновок: Оскільки p >= {alpha}, ми не відхиляємо H0. Залежності не виявлено.")

Статистика Хі-квадрат: 9.9591
p-значення: 0.0016
Висновок: Оскільки p < 0.05, ми відхиляємо H0. Залежність між версією та утриманням є.


Нульова гіпотеза ($H_0$): версія гри та утримання гравця на 7-й день незалежні. Між ними немає зв'язку, різниця в утриманні між gate_30 та gate_40 випадкова.

Альтернативна гіпотеза ($H_a$): версія гри та утримання гравця на 7-й день залежні. Вибір версії гри суттєво впливає на ймовірність того, що гравець повернеться.

$p$-значення $= 0.0016$ значно менше за рівень значущості 5% ($0.05$), отже ми відхиляємо нульову гіпотезу.